# 🗜️ اضغطها | Media Lite Studio

**واجهة جديدة للصوت والفيديو — إعدادات بسيطة مع Advanced عند الحاجة.**

1. شغّل الخلية الأولى لفتح **Media Lite Studio**. أول مرة قد يُطلب منك توصيل Google Drive.
2. اختار نوع الملف وPreset الصوت أو الفيديو. افتح **Advanced Settings** فقط لو محتاج تعدّل Codec أو Bitrate أو الجودة والدقة.
3. اضغط **▶ تشغيل الضغط**. هيتحفظ **آخر إعدادات تشغيل تلقائيًا في Google Drive**، وهتوصلك النتائج في نفس قناة **🗜️ اضغطها | Media Lite** على تيليجرام.
4. اضغط **💾 Save as Default** لتثبيت إعداداتك المفضلة. وزر **استرجاع الافتراضي** يرجّعها لاحقًا من غير ما يبدأ معالجة.
5. إعداداتك محفوظة في \`MyDrive/Telegram_Extreme_Compressor/Media_Lite_Studio/preferences_v1.json\`، منفصلة عن مفاتيح Telegram وجلسة تسجيل الدخول.

**أوامر تيليجرام القديمة شغالة:** Reply بـ\`1\` لإعادة المعالجة بالإعدادات المحفوظة للنتيجة نفسها، \`2\` لأصغر فيديو، \`3\` لصوت صغير جدًا، أو رقم مثل \`80\` لحجم فيديو مستهدف.

> **التوافق:** تقدر تختار تشغيل المحرك القديم من نفس الخلية لو احتجت. خلايا النقل بين Drive وTelegram تحت الواجهة لم تتغير.


In [ ]:
#@title 🗜️ Media Lite Studio — افتح لوحة التحكم
#@markdown **Studio** يحفظ آخر إعدادات تشغيل في Drive ويعرض Advanced بشكل اختياري.
واجهة_التشغيل = "Studio الجديدة (موصى بها)" #@param ["Studio الجديدة (موصى بها)", "تشغيل مباشر بالمحرك القديم"]

import os
import re
import sys
import json
import time
import types
import subprocess
import urllib.request
import urllib.error

REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "telegram-smart-compressor/engine.py"
UI_REL = "telegram-smart-compressor/media_studio_ui.py"
ENGINE_PATH = "/content/telegram_smart_compressor_engine.py"
UI_PATH = "/content/media_studio_ui.py"

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True, text=True, timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={"Accept": "application/vnd.github+json",
                 "User-Agent": "Media-Lite-Studio-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

def download_component(sha, rel):
    req = urllib.request.Request(
        f"https://raw.githubusercontent.com/{REPO}/{sha}/{rel}",
        headers={"User-Agent": "Media-Lite-Studio-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=45) as response:
        return response.read().decode("utf-8")

try:
    print("🔄 تحميل Media Lite Studio...")
    sha = latest_sha()
    os.environ["TSC_ENGINE_SHA"] = sha
    # The UI owns the explicit Run button. Keep the legacy engine from
    # executing automatically when we import its Python functions.
    previous_ui_only = os.environ.get("TSC_UI_ONLY")
    os.environ["TSC_UI_ONLY"] = "1"
    try:
        engine_source = download_component(sha, ENGINE_REL)
        if len(engine_source) < 5000:
            raise RuntimeError("engine-too-small")
        engine = types.ModuleType("tsc_engine_live")
        engine.__file__ = ENGINE_PATH
        sys.modules["tsc_engine_live"] = engine
        exec(compile(engine_source, ENGINE_PATH, "exec"), engine.__dict__)
    finally:
        if previous_ui_only is None:
            os.environ.pop("TSC_UI_ONLY", None)
        else:
            os.environ["TSC_UI_ONLY"] = previous_ui_only
    print(f"✅ Media Lite Engine {engine.ENGINE_BUNDLE_VERSION} — commit {sha[:10]}")

    if واجهة_التشغيل.startswith("Studio"):
        ui_source = download_component(sha, UI_REL)
        if len(ui_source) < 6000:
            raise RuntimeError("studio-ui-incomplete")
        ui = types.ModuleType("tsc_studio_ui")
        ui.__file__ = UI_PATH
        sys.modules["tsc_studio_ui"] = ui
        exec(compile(ui_source, UI_PATH, "exec"), ui.__dict__)
        ui.show_studio(engine)
    else:
        print("↩️ تشغيل الإعداد التلقائي القديم. لتعديل الجودة افتح Studio الجديدة.")
        os.environ["TSC_PROFILE"] = "SMART_AUTO"
        os.environ["TSC_TARGET_SIZE_MB"] = ""
        engine.ACTIVE_STUDIO_SETTINGS = None
        engine.PROFILE = "SMART_AUTO"
        engine.TARGET_SIZE_TEXT = ""
        engine.run()
except urllib.error.HTTPError as exc:
    print(f"❌ E001 — تعذر الوصول إلى GitHub (HTTP {exc.code}). جرّب مرة أخرى.")
except urllib.error.URLError:
    print("❌ E001 — تعذر الاتصال بـ GitHub. تأكد من الإنترنت وجرب مرة أخرى.")
except SyntaxError:
    print("❌ E002 — كود المحرك أو الواجهة فيه خطأ صياغة. راجع نسخة GitHub.")
except Exception as exc:
    print("❌ E003 — تعذر بدء Media Lite Studio.")
    print(f"التفاصيل: {type(exc).__name__}: {str(exc)[:240]}")


## ☁️ 1 — من Google Drive إلى Telegram

**عايز تبعت فولدر أو ملف من Drive إلى تيليجرام؟** استخدم زر التشغيل تحت العنوان ده فقط.

الصق **رابط الملف أو الفولدر** في خانة **رابط Drive**. القناة الخاصة باسم **📁 Media Lite | File Transfer** هتتعمل تلقائيًا لو مش موجودة، أو يستخدم القناة الحالية.

سيب **عدد الملفات = 0** علشان ينقل كل الملفات، وفَعّل المجلدات الفرعية لو عايز محتويات الفولدر كاملة. جرب **معاينة فقط** الأول، وبعدها غيّر إلى **تنفيذ النقل**.

**تسجيل الدخول أول مرة:** اختر **QR بدون كود** لو رسالة تسجيل الدخول مش واصلة. هيظهر QR في Colab، افتح Telegram → الإعدادات → الأجهزة → ربط جهاز على جهاز مسجل دخول وامسحه. لو Colab وTelegram على نفس الجهاز، جرّب رابط الموافقة اللي هيظهر تحت الرمز. تقدر تختار **كود داخل Telegram** لو الكود بيوصلك في المحادثة الرسمية. بعد النجاح الجلسة بتتحفظ على Drive.

**إعادة رفع ملفات مختارة بدون تكرار الباقي:** شغّل **معاينة فقط**؛ هيظهر لكل ملف رقم وحالة «مكتمل/جديد». اكتب أرقام الملفات المطلوب إعادة رفعها في خانة **أرقام الملفات لإعادة الرفع**، مثل `1-7` أو `1,3,5-8`، واختار **إعادة رفع المحدد**. هتترفع الأرقام دي فقط حتى لو كانت مكتملة قبل كده، والأصل على Drive والرسائل القديمة في Telegram هيفضلوا زي ما هم. المحاولة بتتسجل علشان تقدر تكملها لو الاتصال انقطع.

> لو محتويات الفولدر اتغيرت بعد المعاينة، اعمل معاينة جديدة قبل استخدام الأرقام. خيار **تنفيذ النقل** المعتاد بيفضل ينقل الملفات الجديدة فقط ولا يكرر المكتمل.


In [ ]:
#@title ☁️ تشغيل: Drive → Telegram
#@markdown **رابط Drive:** رابط ملف أو فولدر هيتنقل لقناة Telegram الخاصة.
رابط_Drive = "" #@param {type:"string"}
#@markdown **0 = جميع الملفات الجديدة**. اختار رقم أكبر لو عايز دفعة صغيرة.
عدد_الملفات = 0 #@param {type:"integer"}
#@markdown **إعادة رفع ملفات محددة:** شغّل «معاينة فقط» الأول عشان تعرف أرقام الملفات، ثم اكتب مثلًا 1-7 أو 1,3,5-8 هنا.
ارقام_الملفات_لإعادة_الرفع = "" #@param {type:"string"}
شامل_المجلدات_الفرعية = True #@param {type:"boolean"}
#@markdown **أول دخول Telegram:** لو الكود مش بيوصلك، استخدم QR ووافق من تطبيق Telegram.
طريقة_الدخول = "QR بدون كود" #@param ["QR بدون كود", "كود داخل Telegram"]
التشغيل = "معاينة فقط" #@param ["معاينة فقط", "تنفيذ النقل", "إعادة رفع المحدد"]

import json
import re
import subprocess
import time
import urllib.request
import types

_TRANSFER_REPO = "abdullahsamirashour/gpt"
_TRANSFER_BRANCH = "main"
_TRANSFER_ENGINE = "telegram-smart-compressor/transfer_engine.py"

def _ml_transfer_latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_TRANSFER_REPO}.git",
         f"refs/heads/{_TRANSFER_BRANCH}"],
        capture_output=True, text=True, timeout=30,
    )
    sha = p.stdout.strip().split()[0] if p.returncode == 0 and p.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", sha):
        return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_TRANSFER_REPO}/git/ref/heads/{_TRANSFER_BRANCH}?v={int(time.time())}",
        headers={"User-Agent": "Media-Lite-Transfer-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

print("🔄 تحميل أحدث محرك نقل من GitHub...")
_transfer_sha = _ml_transfer_latest_sha()
_transfer_url = f"https://raw.githubusercontent.com/{_TRANSFER_REPO}/{_transfer_sha}/{_TRANSFER_ENGINE}"
with urllib.request.urlopen(
    urllib.request.Request(_transfer_url, headers={"User-Agent": "Media-Lite-Transfer-Colab"}),
    timeout=45,
) as _response:
    _transfer_code = _response.read().decode("utf-8")
if len(_transfer_code) < 10000:
    raise RuntimeError("محرك النقل من GitHub غير مكتمل.")
_transfer_module = types.ModuleType("media_lite_transfer")
exec(compile(_transfer_code, _TRANSFER_ENGINE, "exec"), _transfer_module.__dict__)
print(f"✅ المحرك جاهز — commit {_transfer_sha[:10]}")
_transfer_module.run_transfer(
    direction="drive_to_telegram",
    source=رابط_Drive.strip(),
    destination="",
    max_files=عدد_الملفات,
    recursive=شامل_المجلدات_الفرعية,
    preview=التشغيل == "معاينة فقط",
    login_method="qr" if طريقة_الدخول == "QR بدون كود" else "code",
    reupload_selected=التشغيل == "إعادة رفع المحدد",
    selected_numbers=ارقام_الملفات_لإعادة_الرفع.strip(),
)


## 📥 2 — من Telegram إلى Google Drive

**عايز تحفظ ملفات قناة تيليجرام على Drive؟** استخدم زر التشغيل تحت العنوان ده فقط.

**قناة Telegram المصدر:** لو فاضية هيستخدم قناة **📁 Media Lite | File Transfer**. لو عايز قناة تانية، اكتب `@username` أو معرف القناة المتاح لحسابك.

**فولدر Drive للحفظ:** لو فاضي هينشئ أو يستخدم مجلد **Media Lite Transfers**. لو عندك مجلد معين، الصق رابطه.

سيب **عدد الملفات = 0** للكل، وجرب المعاينة الأول. الملفات الأصلية مش بتتمسح أو تتضغط.

**تسجيل الدخول أول مرة:** اختر **QR بدون كود** لو رسالة تسجيل الدخول مش واصلة. هيظهر QR في Colab، افتح Telegram → الإعدادات → الأجهزة → ربط جهاز على جهاز مسجل دخول وامسحه. لو Colab وTelegram على نفس الجهاز، جرّب رابط الموافقة اللي هيظهر تحت الرمز. تقدر تختار **كود داخل Telegram** لو الكود بيوصلك في المحادثة الرسمية. بعد النجاح الجلسة بتتحفظ على Drive.


In [ ]:
#@title 📥 تشغيل: Telegram → Drive
#@markdown **قناة المصدر:** اتركها فاضية لقناتنا الخاصة أو اكتب @username لقناة أخرى.
قناة_Telegram_المصدر = "" #@param {type:"string"}
#@markdown **فولدر Drive:** اتركه فاضي لمجلد Media Lite Transfers أو الصق رابط فولدر.
فولدر_Drive_للحفظ = "" #@param {type:"string"}
#@markdown **0 = جميع الملفات الجديدة**.
عدد_الملفات = 0 #@param {type:"integer"}
#@markdown **أول دخول Telegram:** لو الكود مش بيوصلك، استخدم QR ووافق من تطبيق Telegram.
طريقة_الدخول = "QR بدون كود" #@param ["QR بدون كود", "كود داخل Telegram"]
التشغيل = "معاينة فقط" #@param ["معاينة فقط", "تنفيذ النقل"]

import json
import re
import subprocess
import time
import urllib.request
import types

_TRANSFER_REPO = "abdullahsamirashour/gpt"
_TRANSFER_BRANCH = "main"
_TRANSFER_ENGINE = "telegram-smart-compressor/transfer_engine.py"

def _ml_transfer_latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_TRANSFER_REPO}.git",
         f"refs/heads/{_TRANSFER_BRANCH}"],
        capture_output=True, text=True, timeout=30,
    )
    sha = p.stdout.strip().split()[0] if p.returncode == 0 and p.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", sha):
        return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_TRANSFER_REPO}/git/ref/heads/{_TRANSFER_BRANCH}?v={int(time.time())}",
        headers={"User-Agent": "Media-Lite-Transfer-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

print("🔄 تحميل أحدث محرك نقل من GitHub...")
_transfer_sha = _ml_transfer_latest_sha()
_transfer_url = f"https://raw.githubusercontent.com/{_TRANSFER_REPO}/{_transfer_sha}/{_TRANSFER_ENGINE}"
with urllib.request.urlopen(
    urllib.request.Request(_transfer_url, headers={"User-Agent": "Media-Lite-Transfer-Colab"}),
    timeout=45,
) as _response:
    _transfer_code = _response.read().decode("utf-8")
if len(_transfer_code) < 10000:
    raise RuntimeError("محرك النقل من GitHub غير مكتمل.")
_transfer_module = types.ModuleType("media_lite_transfer")
exec(compile(_transfer_code, _TRANSFER_ENGINE, "exec"), _transfer_module.__dict__)
print(f"✅ المحرك جاهز — commit {_transfer_sha[:10]}")
_transfer_module.run_transfer(
    direction="telegram_to_drive",
    source=قناة_Telegram_المصدر.strip(),
    destination=فولدر_Drive_للحفظ.strip(),
    max_files=عدد_الملفات,
    recursive=True,
    preview=التشغيل == "معاينة فقط",
    login_method="qr" if طريقة_الدخول == "QR بدون كود" else "code",
)
